In [1]:
import os
import json
import shutil
import numpy as np
import pandas as pd
from collections import defaultdict

In [3]:
# Load sentence meta data and phone mapping
transcription_meta_path = (
    "/data/data/data/xingyuan/animal_language/v2/out/fairseq/"
    "six_balance/work/iter2_130k_11th_km250/metadata_trans.json"
)
replacement_file = (
    "/home/theron/ALP/AniLing/PhonemeFinding/checkpoints/"
    "phoneme_refined/241013_001019/mapping_phoneme.json"
)
with open(transcription_meta_path, "r") as f:
    transcription_meta = json.load(f)
with open(replacement_file, "r") as f:
    replacement = json.load(f)

In [4]:
def replace_phones(word: str, replacement: dict):
    """
    Replace the phones in a word with the given replacement.
    """
    word_phone_list = word.split(" ")
    new_phone_list = []

    for phone in word_phone_list:
        new_phone = phone
        while new_phone in replacement:
            new_phone = replacement[new_phone]

        if new_phone_list:
            if new_phone_list[-1] == new_phone:
                continue
            else:
                new_phone_list.append(new_phone)
        else:
            new_phone_list.append(new_phone)

    new_word = " ".join(new_phone_list)
    return new_word

In [5]:
def compress_transcription(transcription: list):
    """
    Compress the transcription by removing the repeating phones.
    """
    new_transcription = []
    for phone in transcription:
        if not new_transcription:
            new_transcription.append(phone)
        else:
            if new_transcription[-1] == phone:
                continue
            else:
                new_transcription.append(phone)
    return new_transcription

def test_compress_transcription():
    test_cases = [
        (["a", "a", "b", "b", "c", "c"], ["a", "b", "c"]),
        (["a", "b", "b", "c", "c"], ["a", "b", "c"]),
        (["a", "a", "a", "a", "a"], ["a"]),
        (["a", "b", "c", "d", "e"], ["a", "b", "c", "d", "e"]),
    ]
    for test_case, expected in test_cases:
        assert compress_transcription(test_case) == expected
    print("compress_transcription passed")
test_compress_transcription()

compress_transcription passed


In [7]:
meta_file = "/data/data/data/xingyuan/animal_language/v2/out/fairseq/six_balance/work/iter2_130k_11th_km250/metadata.jsonlines"
data = []

with open(meta_file, 'r') as f:
    for line in f:
        data.append(json.loads(line))
print(data[0])

for dd in data:
    name = dd["name"]
    transcription = np.load("/data/data" + dd["label_path"])
    compressed_transcription = compress_transcription(transcription)
    recorded_transcription = transcription_meta[name]["transcription"]
    assert compressed_transcription == recorded_transcription
print("All transcriptions are correct")

{'audio_path': '/data/xingyuan/animal_language/v2/out/second_cut/beats-sed_109-6160_6713/chihuahua_thr0.05_offset1_0.5_audios_16k/UCvHJzqB1KZMpZniAVWy3tkA+SYSSAjRjwMI+00000+00001.wav', 'breeds': ['chihuahua'], 'feature_path': '/data/xingyuan/animal_language/v2/out/fairseq/six_balance/work/iter2_130k_11th_km250/feature/UCvHJzqB1KZMpZniAVWy3tkA+SYSSAjRjwMI+00000+00001.npy', 'label_path': '/data/xingyuan/animal_language/v2/out/fairseq/six_balance/work/iter2_130k_11th_km250/label/UCvHJzqB1KZMpZniAVWy3tkA+SYSSAjRjwMI+00000+00001.npy', 'duration': 0.256, 'name': 'UCvHJzqB1KZMpZniAVWy3tkA+SYSSAjRjwMI+00000+00001'}


KeyboardInterrupt: 

In [37]:
mismatch = 0
for sentence_id, meta in transcription_meta.items():
    if "word_intervals" in meta and "parsed_transcription" in meta:
        num_words = sum([1 for word in meta["parsed_transcription"] if word.startswith("w")])
        if num_words != len(meta["word_intervals"]):
            mismatch += 1
            print("Mismatched number of words and word intervals")
print(f"Number of mismatched word intervals: {mismatch}")

Mismatched number of words and word intervals
Mismatched number of words and word intervals
Mismatched number of words and word intervals
Number of mismatched word intervals: 3


In [42]:
no_word_intervals = 0
no_parsed_transcription = 0
for sentence_id, meta in transcription_meta.items():
    if "word_intervals" not in meta:
        no_word_intervals += 1
    if "parsed_transcription" not in meta:
        no_parsed_transcription += 1
print(f"Number of sentences without word intervals: {no_word_intervals}")
print(f"Number of sentences without parsed transcription: {no_parsed_transcription}")

Number of sentences without word intervals: 2
Number of sentences without parsed transcription: 2


In [8]:
# make word statistics
word_durations = defaultdict(list)
word_meta = defaultdict(dict)
for sentence_id, meta in transcription_meta.items():
    # Pass the incomplete words
    if "word_intervals" not in meta:
        continue
    if "parsed_transcription" not in meta:
        continue
    num_words = sum([1 for word in meta["parsed_transcription"] if word.startswith("w")])
    if num_words != len(meta["word_intervals"]):
        continue
    
    word_idx = 0
    for word in meta["parsed_transcription"]:
        if not word.startswith("w"):
            continue
        word_intervals = meta["word_intervals"][word_idx]
        word_idx += 1
        word_start = word_intervals[0]
        word_end = word_intervals[1]
        word_duration = word_end - word_start
        word_audio_path = meta["audio_path"]

        new_word = replace_phones(word[1:], replacement)
        
        if new_word not in word_meta:
            # initialize the word meta
            word_meta[new_word] = {
                "count": 0,
                "samples": [],
                "duratioin_mean": 0,
                "duration_std": 0,
                "duration_max": 0,
                "duration_min": 0,
            }
        word_meta[new_word]["count"] += 1
        word_meta[new_word]["samples"].append({
            "audio_path": word_audio_path,
            "duration": word_duration,
            "start_ts": word_start,
            "end_ts": word_end,
        })

# select the words which have more than 9 occurrences
word_meta = {k: v for k, v in word_meta.items() if v["count"] > 9}
# sort the words by the number of occurrences
word_meta = {k: v for k, v in sorted(word_meta.items(), key=lambda item: item[1]["count"], reverse=True)}
# calculate the mean and std of word duration
for k, v in word_meta.items():
    durations = [sample["duration"] for sample in v["samples"]]
    v["duratioin_mean"] = np.mean(durations)
    v["duration_std"] = np.std(durations)
    v["duration_max"] = np.max(durations)
    v["duration_min"] = np.min(durations)
with open("word_meta_330.json", "w") as f:
    json.dump(word_meta, f, indent=4)



In [9]:
import os
from pydub import AudioSegment

def cut_audio(audio_path, start_time, end_time, save_path):
    """
    Cuts a segment from a WAV audio file and saves it to a folder.

    Parameters:
        audio_path (str): Path to the input WAV file.
        start_time (float): Start time in seconds.
        end_time (float): End time in seconds.
        save_path (str): Directory to save the cut audio segment.

    Returns:
        None
    """
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    # Load audio
    audio = AudioSegment.from_wav(audio_path)
    
    # Convert times to milliseconds
    start_ms = int(start_time * 1000)
    end_ms = int(end_time * 1000)
    
    # Slice audio
    cut_segment = audio[start_ms:end_ms]
    
    # Create output filename
    base_name = os.path.splitext(os.path.basename(audio_path))[0]
    output_filename = f"{base_name}_{start_time:.2f}s_to_{end_time:.2f}s.wav"
    output_path = os.path.join(save_path, output_filename)
    
    # Export cut segment
    cut_segment.export(output_path, format="wav")

In [ ]:
words_list_meaning_file = "/home/theron/projects/web_apps/canine/models/categorized_word_meaning.json"
with open(words_list_meaning_file, "r") as f:
    words_list_meaning = json.load(f)

for cutting_word in word_meta.keys():
    

    for sample in word_meta[cutting_word]["samples"][:5]:
        audio_path = "/data/data" + sample["audio_path"]
        start_time = sample["start_ts"]
        end_time = sample["end_ts"]
        save_path = f"/home/theron/ALP/AniLing/PhonemeFinding/finalwordsamples/{cutting_word}"
        cut_audio(audio_path, start_time, end_time, save_path)
print("Cutting done")

FileNotFoundError: [Errno 2] No such file or directory: '/data/xingyuan/animal_language/v2/out/second_cut/beats-sed_109-6160_6713/chihuahua_thr0.05_offset1_0.5_audios_16k/UCx5KzVQRdskP66rBCS31fBg+UnmjDyZerlc+00002+00000.wav'

In [47]:
selected_word_duration_meta_file = (
    "/home/sinong/ALP/AniLing/PhonemeFinding/selected_word_duration_meta.json"
)
all_word_duration_meta_file = (
    "/home/sinong/ALP/AniLing/PhonemeFinding/word_duration_statistic_330.json"
)
with open(selected_word_duration_meta_file, "r") as f:
    selected_word_duration_meta = json.load(f)
with open(all_word_duration_meta_file, "r") as f:
    all_word_duration_meta = json.load(f)

selected_word_duration_meta_330 = {}
for word, word_meta in selected_word_duration_meta.items():
    if word not in all_word_duration_meta:
        print(f"Word {word} not in all word duration meta")
    else:
        selected_word_duration_meta_330[word] = all_word_duration_meta[word]
with open("selected_word_duration_meta_330.json", "w") as f:
    json.dump(selected_word_duration_meta_330, f, indent=4)


Word 44 95 not in all word duration meta
Word 110 95 not in all word duration meta
Word 55 95 not in all word duration meta
Word 154 95 not in all word duration meta
Word 76 95 not in all word duration meta
Word 42 95 not in all word duration meta
Word 95 110 not in all word duration meta
Word 123 95 not in all word duration meta
Word 55 32 not in all word duration meta
Word 15 138 55 not in all word duration meta
Word 46 95 246 189 76 221 3 180 95 not in all word duration meta


In [6]:
# Test data quality
num_mismatch_phones = 0
num_w0 = 0
num_not_count_w0 = 0
for sentence_id, sentence_meta in transcription_meta.items():
    # cnt_miss_parsed_transcription = 0
    # cnt_miss_word_intervals = 0
    # cnt_mismatch_word_intervals = 0
    # if "parsed_transcription" not in sentence_meta:
    #     print(f"Missing parsed transcription for sentence {sentence_id}")
    #     cnt_miss_parsed_transcription += 1
    # if "word_intervals" not in sentence_meta:
    #     print(f"Missing word intervals for sentence {sentence_id}")
    #     cnt_miss_word_intervals += 1
    # if len(sentence_meta["parsed_transcription"]) != len(sentence_meta["word_intervals"]):
    #     this_parsed_transcription = sentence_meta["parsed_transcription"]
    #     this_word_intervals = sentence_meta["word_intervals"]
    #     print(f"Mismatched word intervals for sentence {sentence_id}")
    #     cnt_mismatch_word_intervals += 1
    # this_phones = sentence_meta["transcription"]
    # this_phone_ts = sentence_meta["phone_ts"]
    # if len(this_phones) != len(this_phone_ts):
    #     print(f"Mismatched phone timestamps for sentence {sentence_id}")
    #     num_mismatch_phones += 1
    if "parsed_transcription" in sentence_meta:
        for word in sentence_meta["parsed_transcription"]:
            new_word = "w" + replace_phones(word[1:], replacement)
            if new_word == "w0":
                num_w0 += 1
                if "word_intervals" not in sentence_meta or len(sentence_meta["word_intervals"]) != len(sentence_meta["parsed_transcription"]):
                    num_not_count_w0 += 1

print(f"Number of w0: {num_w0}")
print(f"Number of w0 not counted: {num_not_count_w0}")


Number of w0: 1056814
Number of w0 not counted: 1056811
